# 4. Применение модели к новым данным

Загружаем сохранённую модель и пайплайн из бук 2, применяем к новым данным (transform + predict).

**Куда пишутся результаты:**
- Предсказания → `models/{dataset_name}/predictions/{hash}_predictions.csv`

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import joblib
from ipyfilechooser import FileChooser
from IPython.display import display

# Путь к deps/repo_glm для axiolyze (если пакет не установлен через pip install -e)
_glm_path = Path('../../deps/repo_glm').resolve()
if str(_glm_path) not in sys.path:
    sys.path.insert(0, str(_glm_path))

# Legacy-скрипты: support, model_io, io_utils, glm_analysis
_legacy_path = Path('../../deps/repo_glm/axiolyze/legacy').resolve()
if str(_legacy_path) not in sys.path:
    sys.path.insert(0, str(_legacy_path))

# Трансформеры — из axiolyze
from axiolyze.transformers import GLMSmartDataFilterTransformation

# support, model_io, glm_analysis — из axiolyze/legacy/ (добавлено в sys.path выше)
from support import selection_rule_column
from model_io import loadm
from glm_analysis import predict_with_excel_formula


def read_df_from_sql(query, sql_server_name=None, db_name=None, **kwargs):
    """Загрузка из SQL: vsk_omtl или fallback на sqlalchemy."""
    try:
        from vsk_omtl import read_df_from_sql as custom_read_sql
        return custom_read_sql(query=query, sql_server_name=sql_server_name, db_name=db_name, **kwargs)
    except (ImportError, NameError):
        try:
            from sqlalchemy import create_engine
            cs = kwargs.pop('connection_string', 'sqlite:///:memory:')
            if sql_server_name and db_name:
                cs = f"mssql+pyodbc://@{sql_server_name}/{db_name}?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"
            engine = create_engine(cs)
            try:
                return pd.read_sql(query, con=engine, **kwargs)
            finally:
                engine.dispose()
        except Exception as e:
            raise ValueError(f"Не удалось загрузить данные из SQL: {e}")


def extract_schema_from_pipeline(pipeline):
    """Извлекает data_schema из первого шага (SmartDataFilter)."""
    for step_name, transformer in pipeline.steps:
        if isinstance(transformer, GLMSmartDataFilterTransformation):
            return transformer.get_data_schema()
    return {}

## Выбор модели и загрузка

In [2]:
dataset_name = 'research_transformers'
dataset_dir = Path('models') / dataset_name
if not dataset_dir.exists():
    raise FileNotFoundError(f"Папка датасета не найдена: {dataset_dir}")

available_hashes = set()
for f in dataset_dir.iterdir():
    if f.name.count('_') >= 1:
        available_hashes.add(f.name.split('_')[0])
available_hashes = list(available_hashes)
selected_hashes = selection_rule_column(available_hashes)

In [3]:
selected_hash = selected_hashes[0]

model_dict = loadm(dataset_name, selected_hash)

pipeline_joblib_path = dataset_dir / f"{selected_hash}_pipeline.joblib"
if not pipeline_joblib_path.exists():
    raise FileNotFoundError(
        f"Pipeline не найден: {pipeline_joblib_path}. "
        "Запустите 2_research_transformers_clean и savem."
    )
pipeline = joblib.load(pipeline_joblib_path)

data_schema = extract_schema_from_pipeline(pipeline)
index_columns = data_schema.get('index_columns', [])

## Загрузка данных и предсказание

In [4]:
fc_data = FileChooser(
    path='.', filename='', filter_pattern=('*.csv', '*.feather'),
    title='ВАРИАНТ 1: Выберите файл данных (CSV или Feather)'
)
display(fc_data)

fc_sql = FileChooser(
    path='.', filename='', filter_pattern=('*.sql',),
    title='ВАРИАНТ 2: Выберите файл с SQL запросом'
)
display(fc_sql)

FileChooser(path='C:\Users\puls\PycharmProjects\glm', filename='', title='ВАРИАНТ 1: Выберите файл данных (CSV…

FileChooser(path='C:\Users\puls\PycharmProjects\glm', filename='', title='ВАРИАНТ 2: Выберите файл с SQL запро…

In [5]:
if fc_data.selected:
    fp = Path(fc_data.selected)
    if fp.suffix == '.csv':
        df = pd.read_csv(fp)
    elif fp.suffix == '.feather':
        df = pd.read_feather(fp)
    else:
        df = pd.read_csv(fp)
    print(f"Загружено из файла: {fp}")
elif fc_sql.selected:
    with open(fc_sql.selected, 'r', encoding='utf-8') as f:
        query = f.read()
    df = read_df_from_sql(query=query, sql_server_name=None, db_name=None, n_jobs=1, debug_sql=False)
    print(f"Загружено из SQL: {fc_sql.selected}")
else:
    print("Не выбран источник данных. Выберите файл или SQL запрос выше.")
    df = None

if df is not None:
    print(f"Загружено строк: {df.shape[0]}, колонок: {df.shape[1]}")
    df_prepared = df.copy()
    df_transformed = pipeline.transform(df_prepared).copy()

    # Заполнение пропусков (как в бук 2): категориальные → representative, числовые → mean из модели
    cat_params = model_dict['categorical_params']
    num_params = model_dict.get('numerical_params', {})
    repr_levels = model_dict.get('representative_levels', {})
    for c in cat_params:
        if c in df_transformed.columns and c in repr_levels:
            fill_val = repr_levels[c] if repr_levels[c] in cat_params[c] else cat_params[c][0]
            df_transformed[c] = df_transformed[c].fillna(fill_val)
    for c in num_params:
        if c in df_transformed.columns and 'center' in num_params.get(c, {}):
            df_transformed[c] = df_transformed[c].fillna(num_params[c]['center'])

    predictions, _ = predict_with_excel_formula(model_dict, df_transformed)
    df_transformed = df_transformed.copy()
    df_transformed['predicted'] = predictions.flatten()

    df['predicted'] = df_transformed['predicted']

    # Сохранение результатов
    output_dir = dataset_dir / 'predictions'
    output_dir.mkdir(parents=True, exist_ok=True)
    out_path = output_dir / f"{selected_hash}_predictions.csv"
    df.to_csv(out_path, index=False)
    print(f"Предсказания сохранены: {out_path}")

    df

Загружено из файла: C:\Users\puls\PycharmProjects\glm\unbalanced_dataset_train.csv
Загружено строк: 10000, колонок: 6
Предсказания сохранены: models\research_transformers\predictions\b8b0d9136e9dd73810c41881cbb82553_predictions.csv
